# NASA Exoplanet Archive: Data Collection & Provenance

## 1. Overview & Provenance
This notebook documents and performs data acquisition for the **ExoLens** project from the official **NASA Exoplanet Archive**.

* **Data Source**: NASA Exoplanet Archive (Caltech/IPAC under NASA contract)
* **Table Name**: `pscomppars` (Planetary Systems Composite Parameters)
* **Protocol / Endpoint**: Table Access Protocol (TAP) synchronous query via `https://exoplanetarchive.ipac.caltech.edu/TAP/sync`
* **Query Language**: Astronomical Data Query Language (ADQL)
* **Query Statement**: `SELECT * FROM pscomppars`
* **Output Format**: Comma-Separated Values (CSV)
* **Output Artifact**: `exoplanet_data.csv` (Raw tabular snapshot, 6,445 confirmed exoplanets across 703 columns)

### Architectural Note: API Access vs. Web Scraping
The NASA Exoplanet Archive is accessed via its standardized **IVOA TAP (Table Access Protocol) API** using ADQL, which is the authoritative, documented, and machine-readable method for programmatic retrieval of astronomical catalog data. Traditional HTML screen-scraping (e.g. with BeautifulSoup) is neither suitable nor sanctioned for bulk astronomical tables, as the web portal renders paginated dynamic views of millions of measurement cells while the TAP service exposes the complete scientific database with full precision.

In [1]:
import os
import sys
import datetime
from io import StringIO
from pathlib import Path
import requests
import pandas as pd

print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Python {sys.version.split()[0]} environment ready.")
print("Requests and Pandas libraries loaded successfully.")

[2026-10-29 14:36:12] Python 3.11.9 environment ready.
Requests and Pandas libraries loaded successfully.


## 2. Robust Data Acquisition with Offline Fallback

To ensure reproducibility across evaluation environments (including sandboxed execution, air-gapped systems, and viva presentations), the retrieval pipeline:
1. Detects whether the local raw data snapshot (`exoplanet_data.csv`) is already present.
2. Supports querying the live NASA TAP service when network access is enabled.
3. Validates HTTP response codes and payload sizes before persisting.
4. Falls back cleanly to the verified local archive snapshot if the network is unavailable or times out, preventing execution failure.

In [2]:
# Target configuration
TAP_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
ADQL_QUERY = "SELECT * FROM pscomppars"
OUTPUT_FILE = Path("exoplanet_data.csv")

# Retrieval flag: set to False by default to prioritize verified local snapshot
FETCH_LIVE_DATA = False

df = None

if FETCH_LIVE_DATA:
    print(f"Connecting to NASA TAP service at {TAP_URL}...")
    params = {"query": ADQL_QUERY, "format": "csv"}
    try:
        response = requests.get(TAP_URL, params=params, timeout=60)
        response.raise_for_status()
        payload_mb = len(response.content) / (1024 * 1024)
        print(f"HTTP Status {response.status_code}: Successfully received {payload_mb:.2f} MB payload.")
        
        # Parse into DataFrame
        temp_df = pd.read_csv(StringIO(response.text), low_memory=False)
        if len(temp_df) > 5000 and 'pl_name' in temp_df.columns:
            df = temp_df
            df.to_csv(OUTPUT_FILE, index=False)
            print(f"Updated '{OUTPUT_FILE}' with {len(df):,} records.")
        else:
            print("Validation warning: Live data failed sanity checks. Keeping local file.")
    except Exception as exc:
        print(f"Notice: Live TAP retrieval encountered: {exc}")
        print("Proceeding to local archive snapshot fallback...")

# Fallback: Load verified local raw dataset
if df is None:
    if OUTPUT_FILE.exists():
        file_size_mb = OUTPUT_FILE.stat().st_size / (1024 * 1024)
        print(f"Loading verified local raw snapshot: '{OUTPUT_FILE}' ({file_size_mb:.2f} MB)...")
        df = pd.read_csv(OUTPUT_FILE, low_memory=False)
        print(f"Successfully loaded {len(df):,} confirmed exoplanets across {len(df.columns)} columns.")
    else:
        raise FileNotFoundError(f"Critical Error: Neither live TAP service nor '{OUTPUT_FILE}' is available.")

Loading verified local raw snapshot: 'exoplanet_data.csv' (70.53 MB)...


Successfully loaded 6,445 confirmed exoplanets across 703 columns.


## 3. Data Integrity & Validation Checks

Before passing raw data to downstream preparation, we verify:
* Dataset dimensions and minimum row count.
* Uniqueness of the planetary primary key (`pl_name`).
* Presence of foundational physical, orbital, and stellar columns.

In [3]:
# 1. Row and Column Sanity
assert len(df) >= 5000, f"Expected >= 5000 records, found {len(df)}"
assert len(df.columns) >= 100, f"Expected comprehensive parameter set, found {len(df.columns)} columns"

# 2. Primary Key Uniqueness
duplicate_count = df['pl_name'].duplicated().sum()
assert duplicate_count == 0, f"Found {duplicate_count} duplicate planet names!"

print(f"Data Validation PASSED:")
print(f"  • Confirmed Exoplanet Records: {len(df):,}")
print(f"  • Total Catalog Columns: {len(df.columns):,}")
print(f"  • Unique Planet Identifiers (pl_name): {df['pl_name'].nunique():,} (0 duplicates)")

Data Validation PASSED:
  • Confirmed Exoplanet Records: 6,445
  • Total Catalog Columns: 703
  • Unique Planet Identifiers (pl_name): 6,445 (0 duplicates)


## 4. Key Parameter Inspection

We inspect a preview of the confirmed exoplanets catalog along with fundamental observational and physical properties.

In [4]:
df.head(5)[['pl_name', 'hostname', 'discoverymethod', 'disc_year', 'pl_orbper', 'pl_rade', 'pl_bmasse']]

,pl_name,hostname,discoverymethod,disc_year,pl_orbper,pl_rade,pl_bmasse
0,HD 2039 b,HD 2039,Radial Velocity,2002,1120.000000,12.700000,1999.1507
1,HAT-P-8 b,HAT-P-8,Transit,2008,3.076340,15.692600,406.8224
2,K2-43 b,K2-43,Transit,2016,3.471149,4.510000,18.5000
3,Kepler-1753 b,Kepler-1753,Transit,2021,16.004601,2.226781,5.5900
4,Kepler-1176 b,Kepler-1176,Transit,2016,24.173858,2.450000,6.5700


In [5]:
key_columns = [
    "pl_name", "hostname", "discoverymethod", "disc_year", "disc_facility",
    "pl_orbper", "pl_orbsmax", "pl_rade", "pl_bmasse", "pl_dens", "pl_insol", "pl_eqt",
    "st_teff", "st_rad", "st_mass", "st_met", "sy_dist"
]

available_cols = [c for c in key_columns if c in df.columns]
print(f"Key astrophysical columns identified: {len(available_cols)} / {len(key_columns)}")
df[available_cols].info()

Key astrophysical columns identified: 17 / 17
<class 'pandas.DataFrame'>
RangeIndex: 6445 entries, 0 to 6444
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pl_name          6445 non-null   str    
 1   hostname         6445 non-null   str    
 2   discoverymethod  6445 non-null   str    
 3   disc_year        6445 non-null   int64  
 4   disc_facility    6445 non-null   str    
 5   pl_orbper        6092 non-null   float64
 6   pl_orbsmax       5953 non-null   float64
 7   pl_rade          6395 non-null   float64
 8   pl_bmasse        6414 non-null   float64
 9   pl_dens          6297 non-null   float64
 10  pl_insol         5853 non-null   float64
 11  pl_eqt           5909 non-null   float64
 12  st_teff          6138 non-null   float64
 13  st_rad           6114 non-null   float64
 14  st_mass          6436 non-null   float64
 15  st_met           5747 non-null   float64
 16  sy_dist          6417 non

In [6]:
# Descriptive statistics for numerical key properties
df[available_cols].describe().round(3)

,disc_year,pl_orbper,pl_orbsmax,pl_rade,pl_bmasse,pl_dens,pl_insol,pl_eqt,st_teff,st_rad,st_mass,st_met,sy_dist
count,6445.000,6.092000e+03,5953.000,6395.000,6414.000,6297.000,5853.000,5909.000,6138.000,6114.000,6436.000,5747.000,6417.000
mean,2017.395,7.027348e+04,15.422,5.846,401.339,4.814,533.240,836.633,5389.238,1.482,0.933,0.017,705.230
std,5.198,5.152872e+06,345.542,5.411,1133.324,33.776,9864.927,496.280,1719.835,3.834,0.386,0.189,1230.682
min,1992.000,9.100000e-02,0.004,0.310,0.020,0.005,0.000,1.480,415.000,0.012,0.009,-1.000,1.301
25%,2014.000,4.285000e+00,0.052,1.850,4.320,1.270,10.150,486.000,4893.250,0.770,0.770,-0.080,99.556
50%,2016.000,1.064100e+01,0.103,2.860,9.320,2.510,66.714,758.000,5544.000,0.950,0.940,0.020,349.247
75%,2021.000,3.800000e+01,0.313,11.900,188.076,4.530,284.211,1095.000,5900.000,1.250,1.086,0.130,817.584
max,2026.000,4.020000e+08,19000.000,87.206,9534.852,2000.000,636352.126,7777.780,57000.000,88.475,9.100,0.790,8500.000
